# Retrieval-Augmented Generation (RAG) Pipeline

This notebook demonstrates the complete RAG pipeline.

The pipeline is divided into two major phases:

1. **Document Ingestion and Indexing**
   - Read the document
   - Preprocess the extracted text
   - Create chunks
   - Generate embeddings
   - Store the embeddings in a vector store

2. **Query and Response Generation**
   - Receive a user query
   - Generate the query embedding
   - Retrieve relevant document chunks
   - Build a prompt using the query and retrieved context
   - Generate the final response using an LLM

# Phase 1: Document Ingestion and Indexing

In this phase, the source document is transformed into searchable vector representations.

The document passes through the following pipeline:

PDF
↓
Text Extraction
↓
Text Preprocessing
↓
Chunking
↓
Embedding Generation
↓
Vector Store

## 1. PDF Reading and Text Extraction

The first step is to extract the content from the PDF document.

The document is initially converted into a page-wise representation.

Each element of the resulting list represents the extracted text from one page of the PDF.

This page-wise representation is preserved because the page boundary provides useful structural information for subsequent preprocessing.

In [ ]:
from pypdf import PdfReader

In [ ]:
pdf = []
try:
    reader = PdfReader('./src/pdfs/logical_reasoning.pdf')
    for page in reader.pages:
         pdf.append(page.extract_text())

    print(f"Total Pages: {len(pdf)}")

except Exception as exc:
    # TO Catch error 
    print("Remaining")

In [ ]:
page = pdf[8]
page

## 2. Text Preprocessing

The raw text extracted from a PDF may contain structural noise such as:

- Repeated headers
- Repeated footers
- Page numbers
- Other boilerplate content

The purpose of this stage is to identify and remove repeated structural content while preserving the actual semantic content of the document.

The preprocessing pipeline uses similarity analysis and statistical patterns to identify boilerplate content dynamically rather than relying on document-specific hardcoded strings.

In [ ]:
from src.preprocessing.cleaner import normalize_pdf
from src.preprocessing.similarity import compute_line_similarity
from src.preprocessing.boilerplate import identify_boilerplate, remove_boilerplate

In [ ]:
# Phase 1: PDF Text normalization
normalized_pdf = normalize_pdf(pdf)

# # Phase 2: Statistical Analysis
similar_line_analysis = compute_line_similarity(pdf=normalized_pdf, candidate_lines=3)

# Phase 3: Boilerplate Identification
candidate_boilerplates = identify_boilerplate(similarity_df=similar_line_analysis)

# print(candidate_boilerplates)

# # Phase 4: Boilerplate Removal
boilerplate_config = (
    candidate_boilerplates[
        candidate_boilerplates["is_boilerplate"]
    ][["region", "line"]]
)

filtered_pdf = remove_boilerplate(normalized_pdf, boilerplate_config=boilerplate_config)

# Phase 5: Converting page-wise PDF into the one combined text
text = "".join(filtered_pdf)

## 3. Text Chunking

The cleaned document is divided into smaller text segments called chunks.

Chunking is necessary because the entire document cannot be efficiently embedded and retrieved as a single unit.

The objective of chunking is to create meaningful units of text that:

- Preserve sufficient semantic context
- Are small enough for efficient embedding and retrieval
- Can be independently retrieved in response to a user query

In [ ]:
# from src.chunking.chunker import text_splitter

In [ ]:
# chunks = text_splitter(text=text)


# for i, chunk in enumerate(chunks):
#     print(f"\n--- Chunk {i+1} ---")
#     print(f"\n--- First half of chunk {i+1} ---")
#     print(chunk[:300])
#     print(f"\n--- Second half of chunk {i+1} ---")
#     print(chunk[300:])
#     print(f"\nLength: {len(chunk)}")

#     if i == 2:
#         break

# Experiment: Detecting Semantic Boundaries Using Adjacent Text Similarity

## Objective

Investigate whether a transition between topics can be detected by measuring the semantic similarity between adjacent text units.

The hypothesis is:

> Text units belonging to the same topic should generally have higher semantic similarity, while a transition between topics may produce a noticeable decrease in similarity.

The goal of this experiment is **not yet to build the final chunker**.

Instead, we want to determine whether semantic similarity provides a useful signal for identifying potential topic boundaries.

---

## Experiment Pipeline

```text
Raw Document
      ↓
Text Units
      ↓
Embedding Generation
      ↓
Adjacent Similarity Calculation
      ↓
Semantic Boundary Detection
      ↓
Semantic Groups
      ↓
RecursiveCharacterTextSplitter
      ↓
Final Retrieval Chunks

In [ ]:
import re

# Generic numbering:
# 1 Introduction
# 1. Introduction
# 1.2 Bayes Theorem
# IV Probability

# NUMBERING_PATTERN = re.compile(
#     r"^(\d+(\.\d+)*|[IVXLCDM]+)\.?\s+\S+"
# )

ENUMERATION_PATTERN = re.compile(
    r"""
    ^
    (
        \d+(?:\.\d+)*\.?      |   # 1 / 1.2 / 2.3.1
        [A-Z]\.?              |   # A / A.
        [A-Z]\)               |   # A)
        [IVXLCDM]+\.?             # I / II / IV.
    )
    \s+
    \S+
    """,
    re.VERBOSE,
)


def heading_score(block: str) -> int:
    """
    Compute a structural score indicating how likely a block is a heading.
    """

    score = 0

    block = block.strip()

    if not block:
        return 0

    words = block.split()
    word_count = len(words)
    line_count = len(block.splitlines())

    # -------------------------------------------------
    # Rule 1: Numbering
    # -------------------------------------------------
    if ENUMERATION_PATTERN.match(block):
        score += 4

    # -------------------------------------------------
    # Rule 2: Ends without sentence punctuation
    # -------------------------------------------------
    if not block.endswith((".", "!", "?", ";", ":")):
        score += 2

    # -------------------------------------------------
    # Rule 3: Word count
    # -------------------------------------------------
    if 1 <= word_count <= 10:
        score += 2

    # -------------------------------------------------
    # Rule 4: Line count
    # -------------------------------------------------
    if line_count <= 2:
        score += 2

    # -------------------------------------------------
    # Rule 5: Capitalization ratio
    # -------------------------------------------------
    capitalized = sum(
        1
        for word in words
        if word[:1].isupper()
    )

    capitalization_ratio = capitalized / word_count

    if capitalization_ratio >= 0.7:
        score += 2

    # -------------------------------------------------
    # Rule 6: Sentence-ending punctuation count
    # -------------------------------------------------
    sentence_endings = len(
        re.findall(r"[.!?]", block)
    )

    if sentence_endings == 0:
        score += 2

    # -------------------------------------------------
    # Rule 7: Average word length
    # -------------------------------------------------
    average_word_length = (
        sum(len(word) for word in words)
        / word_count
    )

    if average_word_length >= 5:
        score += 1

    return score


def is_heading(block: str, threshold: int = 10) -> bool:
    """
    Returns True if the block is likely a heading.
    """
    return heading_score(block) >= threshold

In [ ]:
# Identify Lists
import re

LIST_PATTERN = re.compile(
    r"""
    ^
    (
        [-*•▪◦]                    |   # Bullet
        \d+[\.\)]                  |   # 1.  1)
        [A-Za-z][\.\)]             |   # a)  B.
        [IVXLCDM]+[\.\)]               # I.  IV)
    )
    \s+
    """,
    re.VERBOSE,
)


def is_list(block: str) -> bool:
    """
    Returns True if the block looks like a list item.
    """
    return bool(LIST_PATTERN.match(block.strip()))


In [ ]:
# Identify Paragraphs
import re


def is_paragraph(block: str) -> bool:
    """
    Returns True if the block behaves like a paragraph.
    """

    block = block.strip()

    if not block:
        return False

    word_count = len(block.split())

    sentence_count = len(
        re.findall(r"[.!?]", block)
    )

    return (
        word_count >= 20
        and sentence_count >= 1
    )



In [ ]:
from dataclasses import dataclass
from enum import Enum

import re


class BlockType(Enum):
    PARAGRAPH = "paragraph"
    LIST = "list"
    UNKNOWN = "unknown"


@dataclass
class TextUnit:
    type: BlockType
    text: str


LIST_PATTERN = re.compile(
    r"""
    ^
    (
        [-*•▪◦]                  |
        \d+[\.\)]                |
        [A-Za-z][\.\)]           |
        [IVXLCDM]+[\.\)]
    )
    \s+
    """,
    re.VERBOSE,
)


def classify_line(line: str) -> BlockType:
    """
    Classify a single line.
    """

    line = line.strip()

    if not line:
        return None

    if LIST_PATTERN.match(line):
        return BlockType.LIST

    # Everything else is currently unknown.
    # Heading detection will later classify these.
    return BlockType.UNKNOWN


def classify_page(page: str) -> list[TextUnit]:
    """
    Convert a raw PDF page into structured text units.
    Consecutive lines of the same type are merged together.
    """

    blocks = []

    current_type = None
    current_lines = []

    for line in page.splitlines():

        if not line.strip():

            # Blank line ends the current block
            if current_lines:
                blocks.append(
                    TextUnit(
                        type=current_type,
                        text="\n".join(current_lines)
                    )
                )

                current_lines = []
                current_type = None

            continue

        line_type = classify_line(line)

        if current_type is None:

            current_type = line_type
            current_lines.append(line)

        elif line_type == current_type:

            current_lines.append(line)

        else:

            blocks.append(
                TextUnit(
                    type=current_type,
                    text="\n".join(current_lines)
                )
            )

            current_type = line_type
            current_lines = [line]

    if current_lines:

        blocks.append(
            TextUnit(
                type=current_type,
                text="\n".join(current_lines)
            )
        )

    return blocks


def classify_document(pdf: list[str]) -> list[list[TextUnit]]:
    """
    Classify every page in the document.
    """

    return [
        classify_page(page)
        for page in pdf
    ]

In [ ]:
classified_pdf = classify_document(filtered_pdf)

print(classified_pdf[8])

[TextUnit(type=<BlockType.LIST: 'list'>, text='B. Types of Problems in Verbal Reasoning '), TextUnit(type=<BlockType.UNKNOWN: 'unknown'>, text="Class-1: Number Series \nLogical reasoning is a critical aspect of various competitive exams, interviews, and cognitive assessments. \nIt tests an individual's ability to use structured thinking to solve complex problems, often under time \nconstraints. Among the many types of logical reasoning questions, number series questions are \nparticularly common. These questions assess a person's ability to identify patterns, sequences, and \nrelationships between numbers. \nUnderstanding number series and its types is essential for anyone looking to excel in logical reasoning \ntests. This article delves into the intricacies of number series, exploring its significance, different types, \nand strategies for solving them effectively. "), TextUnit(type=<BlockType.LIST: 'list'>, text='a) Understanding Number Series '), TextUnit(type=<BlockType.UNKNOWN: '

## 4. Document Embedding Generation

Each text chunk is converted into a numerical vector representation called an embedding.

The embedding represents the semantic meaning of the chunk in a vector space.

Semantically similar chunks should have similar vector representations.

The result of this stage is:

Text Chunks
        ->
Embedding Model
        ->
Vector Representations

In [ ]:
from src.embeddings.embedder import chunk_encoder

In [ ]:
embeddings = chunk_encoder(chunks)

embeddings

## 5. Vector Store

The generated embeddings are stored in a vector index.

The vector store is responsible for:

- Storing document embeddings
- Maintaining the relationship between vectors and their original text chunks
- Searching for vectors that are most similar to a query embedding

In this project, FAISS is used as the vector search backend.

The document ingestion pipeline is now complete:

PDF
->
Text Extraction
->
Preprocessing
->
Chunking
->
Embedding
->
Vector Store

In [ ]:
from src.vector_store.vector_store import FaissVectorStore

In [ ]:
# Configure FaissVectorStore
dimension = embeddings.shape[1]

vectorStore = FaissVectorStore(dimension=dimension)

In [ ]:
# Adding Embeddings Into the Vector Store
vectorStore.add(embeddings=embeddings)

In [ ]:
# Checking total Indexes
vectorStore.total_vectors()

# Phase 2: Query and Response Generation

The second phase begins when a user submits a question.

The query is transformed into the same vector space as the document chunks so that semantically relevant information can be retrieved.

The pipeline is:

User Query
->
Query Embedding
->
Vector Search
->
Relevant Context
->
Prompt Construction
->
LLM Response Generation

## 1. User Query

The user provides a natural-language question.

The query represents the information that the user wants to retrieve from the indexed document.

In [ ]:
# query = "What is qualitative data analysis?"
query = "Steps to effective quantitative data analysis "


## 2. Query Embedding

The user query is converted into an embedding using the same embedding model used for the document chunks.

Using the same embedding model is important because both the query and the document chunks must exist in the same vector space for meaningful similarity comparison.

In [ ]:
from src.embeddings.embedder import chunk_encoder

In [ ]:
query_embedding = chunk_encoder([query])

## 3. Similarity Search

The query embedding is passed to the vector store.

The vector store compares the query vector with the stored document vectors and retrieves the most semantically similar chunks.

The result of this stage is the relevant context required to answer the user's question.

In [ ]:
# Querying the Vector store
distance, indices = vectorStore.search(query_embedding, 3)

In [ ]:
# Preparing the Retrieved text List
retrieved_text = []
for idx in indices[0]:
    retrieved_text.append(chunks[idx])

retrieved_text

## 4. Prompt Construction and Response Generation

The retrieved document context and the original user query are combined to construct a prompt for the Large Language Model.

The LLM uses the retrieved context to generate the final response.

The final stage is:

User Query
        +
Retrieved Context
        ->
Prompt
        ->
Large Language Model
        ->
Generated Response

In [ ]:
from src.generation.llm import LLMResponseGenerator

In [ ]:
# LLM Configuration
llm =  LLMResponseGenerator()

In [ ]:
# Prompt Building
prompt = llm.prompt_builder(context=retrieved_text, query=query)

In [ ]:
# Response Generation
response = llm.generate_response(prompt=prompt)

In [ ]:
print(response.text)